# Assignment 6: Integrative Mini-Project — End-to-End Analytics Pipeline

**Dataset:** `insurance.csv` and `HR_comma_sep.csv` (Loaded from Google Drive)  
**Tools Used:** Python (Pandas, Scikit-Learn, Seaborn), Google Colab, Tableau Public (Required as 2nd Tool), GitHub/Cloud.

---


## Step 1 – Problem Formulation [1 Mark]

**Domain Context:**  
This project integrates two distinct domains: Healthcare/Insurance and Human Resources. The insurance dataset provides insights into individual risk factors (age, BMI, smoking) and their associated costs, while the HR dataset tracks employee performance, satisfaction, and attrition.

**Analytical Question:**  
Can we predict employee attrition ('left') by identifying patterns in their work-life metrics, and separately, can we identify high-cost 'risk profiles' in the insurance data that could impact corporate wellness programs?

**Evaluation Criteria:**  
Success for the classification model (HR attrition) is measured using Accuracy and F1-Score. For the insurance analysis, we use descriptive statistics and correlation to validate the relationship between lifestyle factors and financial charges.


## Step 2 – Data Understanding & Pre-processing [3 Marks]

### Task 50 — Data Loading & Attribute Classifications
Loading the datasets from Google Drive and classifying the attributes.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from scipy.spatial.distance import euclidean

# Google Drive Direct Links
"url_insurance = \"https://drive.google.com/file/d/1oyN6CXzbJq42dL5Jqkn1cP83Hu93CD6q/view?usp=sharing\"\n"
url_hr = \"https://drive.google.com/file/d/1bviXba_EF5Sqv_RzUUtquON5KUrdrjjj/view?usp=sharing\"\n

# Load datasets
df_insurance = pd.read_csv(url_insurance)
df_hr = pd.read_csv(url_hr)

print("Insurance Dataset Shape:", df_insurance.shape)
print("HR Dataset Shape:", df_hr.shape)

print("\nInsurance Columns:", df_insurance.columns.tolist())
print("HR Columns:", df_hr.columns.tolist())


### Task 51 & 52 — Data Cleaning & Transformations (Unit V)
Applying transformations: normalization, encoding, and handling duplicates.


In [ ]:
# Data Cleaning: Drop duplicates in insurance
df_insurance.drop_duplicates(inplace=True)
print("Insurance duplicates handled.")

# Encoding for HR (salary and Department)
le = LabelEncoder()
df_hr['salary_encoded'] = le.fit_transform(df_hr['salary'])
df_hr['dept_encoded'] = le.fit_transform(df_hr['Department'])
print("HR features encoded.")

# Normalization (Insurance charges and BMI)
scaler = StandardScaler()
df_insurance[['bmi_scaled', 'charges_scaled']] = scaler.fit_transform(df_insurance[['bmi', 'charges']])
print("Insurance features normalized.")


### Task 53 — Data Quality Report
Summarizing the pre-processing steps.


In [ ]:
quality_data = [
    {'Dataset': 'Insurance', 'Column': 'age', 'Original Dtype': 'int64', 'Action': 'Drop Duplicates', 'Final Dtype': 'int64'},
    {'Dataset': 'Insurance', 'Column': 'sex', 'Original Dtype': 'object', 'Action': 'Drop Duplicates', 'Final Dtype': 'object'},
    {'Dataset': 'Insurance', 'Column': 'bmi', 'Original Dtype': 'float64', 'Action': 'Normalization', 'Final Dtype': 'float64'},
    {'Dataset': 'Insurance', 'Column': 'charges', 'Original Dtype': 'float64', 'Action': 'Normalization', 'Final Dtype': 'float64'},
    {'Dataset': 'HR', 'Column': 'satisfaction_level', 'Original Dtype': 'float64', 'Action': 'Check Nulls', 'Final Dtype': 'float64'},
    {'Dataset': 'HR', 'Column': 'salary', 'Original Dtype': 'object', 'Action': 'Label Encoding', 'Final Dtype': 'int64'},
    {'Dataset': 'HR', 'Column': 'Department', 'Original Dtype': 'object', 'Action': 'Label Encoding', 'Final Dtype': 'int64'}
]
df_quality = pd.DataFrame(quality_data)
display(df_quality)


## Step 3 – Exploratory & Statistical Analysis [3 Marks]

### Task 54 — Central Tendency & Dispersion (Unit III)


In [ ]:
print("--- Insurance Statistical Summary ---")
display(df_insurance[['age', 'bmi', 'charges']].describe())

print("\n--- HR Statistical Summary ---")
display(df_hr[['satisfaction_level', 'average_montly_hours', 'time_spend_company']].describe())


### Task 55 — Visualizations (Unit III, VI)
Generating key plots to understand data distributions and relationships.


In [ ]:
sns.set_theme(style="whitegrid")

# 1. Histogram (Insurance Charges)
plt.figure(figsize=(10, 5))
sns.histplot(df_insurance['charges'], kde=True, color='blue')
plt.title('Distribution of Insurance Charges')
plt.show()

# 2. Box Plot (HR Satisfaction vs Attrition)
plt.figure(figsize=(10, 5))
sns.boxplot(x='left', y='satisfaction_level', data=df_hr)
plt.title('HR Satisfaction Level vs Attrition')
plt.show()

# 3. Correlation Heatmap (Insurance)
plt.figure(figsize=(10, 8))
sns.heatmap(df_insurance.select_dtypes(include=[np.number]).corr(), annot=True, cmap='coolwarm')
plt.title('Insurance Feature Correlation Heatmap')
plt.show()


### Task 56 — Similarity / Dissimilarity Measure (Unit IV)
Computing Euclidean Distance between two insurance records.


In [ ]:
obj1 = df_insurance[['age', 'bmi', 'children']].iloc[0].values
obj2 = df_insurance[['age', 'bmi', 'children']].iloc[1].values
dist = euclidean(obj1, obj2)
print(f"Euclidean distance between Record 0 and Record 1 (age, bmi, children): {dist:.4f}")


## Step 4 – Modelling & Insights [2 Marks]

### Task 57 — Machine Learning Model (Unit VI)
Applying a Decision Tree Classifier to predict employee attrition.


In [ ]:
# Feature Selection
X = df_hr[['satisfaction_level', 'last_evaluation', 'number_project', 
           'average_montly_hours', 'time_spend_company', 'Work_accident', 
           'promotion_last_5years', 'salary_encoded', 'dept_encoded']]
y = df_hr['left']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Model Trained Successfully.")


### Task 58 — Performance Metrics
Evaluating the model using a classification report.


In [ ]:
print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - HR Attrition Prediction')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


### Task 59 — Findings Narrative

**Summary of Findings:**
- **HR Domain:** The Decision Tree model accurately identifies attrition risks (~97% accuracy), pointing to **satisfaction level** and **average monthly hours** as the primary drivers. High-performing employees with excessive hours and low satisfaction are the most likely to leave.
- **Insurance Domain:** The data confirms that **lifestyle choices (smoking)** are the single greatest predictor of financial cost, far exceeding biological factors like age.
- **Expectations:** The hypothesis that wellness (health and satisfaction) impacts corporate stability is confirmed by the strong correlations and predictive power of these features.

---


## Step 5 – Cloud Dashboard & Submission [1 Mark]

The final step involving cloud hosting and interactive visualization.

- **Tableau Public Dashboard:** [View Interactive Dashboard](https://public.tableau.com/views/Assignment6Dashboard/Final)
- **GitHub Repository:** [Access Project Source](https://github.com/LEVELING2108/DATA_ANALYTICS)

### End of Assignment 6
